# Create Alzheimer's Drug Discovery Foundation (ADDF) Awards

Creates the funded projects of the **Alzheimer's Drug Discovery Foundation** (US) from the
foundation's own portfolio database.

**Prerequisite:** `scripts/local/addf_to_s3.py` (uploads the parquet).

**Source:** "Our Portfolio" (https://www.alzdiscovery.org/research-and-grants/portfolio).
Ladder step 0: the page's own "Export Results" endpoint
(`/research-and-grants/database-export`) returns every project as text, fetched once for
Active (118) and once for Closed (706; the page header says 707, the export carries 706).
Each project's detail page (`portfolio-details/<id>`) adds the project duration, therapeutic
stage / type, biomarker type, state + country and a lay summary. alzdiscovery.org serves no
robots.txt.

**824 projects, awarded 1999-2026**, USD 412M: Drug Discovery 431, Clinical 107,
Biomarkers 101, Diagnostics Accelerator 89, Conferences/Other 70, Prevention 26.

**Scope (kept and flagged):** the portfolio mixes grants with ADDF's "mission-related
investments" in companies (184 projects go to Biotechnology/For Profit organisations; the
site does not say which were investments), 70 Conferences/Other sponsorships
(`funding_type = 'conference'`, meetings and workshops, USD 2.5k-100k) and the annual Goodes
Prize (`prize`). All are drug-development research funding pathways of a research funder.

**Field mapping:** `display_name` = project title; `description` = lay summary;
`funder_scheme` = Program; `amount` = Funding Amount (USD); `start_date` / `end_date` =
Project Duration; `start_year` = year of start_date, else the Year Awarded;
`lead_investigator` = investigator (name split in the script, degrees dropped), affiliation =
organisation + country (US state codes mapped to United States); 6 consortium-member rows
with no investigator carry the organisation only.

**`funder_award_id` (2.1.1):** ADDF publishes no grant numbers (citing works quote internal
numbers such as `201809-2016862`, `GC-201711-2013978` or `20150701.01` that never appear on
the site), so the key is the synthetic `ADDF-{portfolio id}` (the stable detail-page id).
0 existing citation stubs collapse onto these awards.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320307132`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 0185drb55, doi 10.13039/100002565)

**Priority:** `579` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.addf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/addf/addf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total, COUNT(DISTINCT funder_award_id) as distinct_ids,
       MIN(year_awarded) as min_year, MAX(year_awarded) as max_year,
       ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) as total_usd
FROM openalex.awards.addf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.addf_raw;

In [ ]:
%sql
SELECT program, organization_type, status, COUNT(*) AS n, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS usd
FROM openalex.awards.addf_raw GROUP BY 1, 2, 3 ORDER BY n DESC;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320307132 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320307132;

## Step 2: Create ADDF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.addf_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320307132  -- Alzheimer's Drug Discovery Foundation
),
g AS (
    SELECT *,
           TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS sd,
           TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS ed
    FROM openalex.awards.addf_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(g.title), ''), CONCAT('ADDF grant to ', g.organization)) as display_name,
    NULLIF(TRIM(g.summary), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.program = 'Conferences/Other' THEN 'conference'
         WHEN g.title RLIKE '(?i)goodes prize' THEN 'prize'
         ELSE 'research' END as funding_type,
    NULLIF(TRIM(g.program), '') as funder_scheme,
    'addf_portfolio' as provenance,
    g.sd as start_date,
    g.ed as end_date,
    COALESCE(YEAR(g.sd), TRY_CAST(g.year_awarded AS INT)) as start_year,
    YEAR(g.ed) as end_year,
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.organization), ''),
            'country', NULLIF(TRIM(g.country), ''),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

## Shape check (2.1.1)
Synthetic keys only: `ADDF-{portfolio id}` (digits).

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.addf_awards
WHERE NOT funder_award_id RLIKE '^ADDF-[0-9]{1,10}$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.addf_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'addf_portfolio' AND priority = 579;

-- Priority 579: direct-from-funder ingest of the foundation's own portfolio database.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    579 as priority
FROM openalex.awards.addf_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.addf_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as usd
FROM openalex.awards.addf_awards GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.addf_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.addf_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.addf_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi
FROM openalex.awards.addf_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'addf_portfolio'
GROUP BY provenance, priority;